# 01 — Westminster STATS19 + road network exploration

Quick look at the real data flowing through the Greyspot pipeline before trusting any model result. Run `scripts/run_pipeline.py` at least once first so cached data exists under `data/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from greyspot.ingest.stats19 import load_westminster_collisions

collisions = load_westminster_collisions([2021, 2022, 2023, 2024, 2025], ROOT / "data" / "raw")
collisions[["collision_year", "severity_label", "longitude", "latitude"]].head()

In [ ]:
# Collisions per year and severity mix
collisions.groupby(["collision_year", "severity_label"]).size().unstack(fill_value=0)

In [ ]:
# How many collisions are missing coordinates? (data-quality check per dossier §8)
missing_coords = collisions[["longitude", "latitude"]].isna().any(axis=1).mean()
print(f"{missing_coords:.2%} of Westminster collisions are missing coordinates")

In [ ]:
from greyspot.ingest.network import build_westminster_graph, graph_to_edges_gdf, snap_collisions_to_graph

graph = build_westminster_graph(cache_path=ROOT / "data" / "interim" / "westminster_graph.graphml")
edges = graph_to_edges_gdf(graph)
print(f"{graph.number_of_nodes()} nodes, {len(edges)} edges")
edges[["segment_id", "highway", "length"]].head()

In [ ]:
snapped = snap_collisions_to_graph(collisions, graph)
print(f"Snapped {len(snapped)}/{len(collisions)} collisions to a road segment ({len(snapped)/len(collisions):.1%})")

In [ ]:
# Which segments have the most historical collisions? (sanity check, not a priority score)
from greyspot.features.build_features import collision_counts_by_segment_year

counts = collision_counts_by_segment_year(snapped)
counts.groupby("segment_id")["collision_count"].sum().sort_values(ascending=False).head(10)